# Lab 2 · Dataset Preparation

**~25 minutes.**

Fine-tuning is mostly data work. This notebook covers the part of the deck
that lists dataset formats — except instead of a slide showing four JSON
snippets, you generate all four from the same source and diff them.

> ↳ Slides: *Dataset Format: Raw, Alpaca, ShareGPT, ChatML* · *Apply Chat Template*

In [ ]:
# --- Install the fine-tuning stack --------------------------------------
# Kaggle ships a matched torch/CUDA pair. --no-deps stops pip replacing torch
# with an incompatible build, which shows up later as baffling CUDA errors.
#
# Takes 2-4 minutes. "dependency conflict" warnings here are expected and fine.
# Output is deliberately NOT suppressed: if this step fails, you need to see it.
!pip install -q --no-deps unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub sentencepiece protobuf

print("\ninstall finished - verifying imports...")
import importlib
missing = [m for m in ("unsloth", "trl", "peft", "bitsandbytes", "datasets")
           if importlib.util.find_spec(m) is None]
print("MISSING: " + ", ".join(missing) if missing else "all packages importable")

In [ ]:
# --- Locate the workshop repo -------------------------------------------
# Tries, in order: already present -> attached Kaggle Dataset -> git clone.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR-USERNAME/LLM-lab.git"   # TODO: your repo

def find_repo() -> Path:
    for candidate in [Path("/kaggle/working/LLM-lab"), Path.cwd(), Path.cwd().parent]:
        if (candidate / "common" / "config.py").exists():
            return candidate
    for d in Path("/kaggle/input").glob("*"):          # attached as a Dataset
        if (d / "common" / "config.py").exists():
            return d
    print("Repo not found locally, cloning...")        # last resort
    subprocess.run(["git", "clone", "-q", REPO_URL, "/kaggle/working/LLM-lab"], check=True)
    return Path("/kaggle/working/LLM-lab")

REPO = find_repo()
sys.path[:0] = [str(REPO / "common"), str(REPO / "dataset")]
print(f"repo: {REPO}")

import config
print(config.summary())

## 3.1 The problem we're solving

**AURA** is a fictional HPC cluster. Its facts live in
`dataset/aura_spec.py` — five partitions, four QoS levels, storage
quotas, module versions, wrapper commands like `aura-quota`.

Fictional on purpose. If we fine-tuned on real Slurm documentation, the
base model would already half-know it and the before/after would be a
subtle matter of taste. With AURA, the model starts at zero and the
improvement is unambiguous.

In [ ]:
import aura_spec

print(f"{aura_spec.CLUSTER['full_name']}\n")
for p in aura_spec.PARTITIONS:
    gpus = p["gpus_per_node"] if p["gpus_per_node"] != "none" else "CPU only"
    print(f"  {p['name']:<16} {gpus:<24} max {p['max_walltime_human']}")

## 3.2 Generate the raw conversations

`generate_raw.py` turns the fact file into conversations by asking each
fact many different ways. This is synthetic data generation from a
knowledge base — a genuinely useful technique when you have documentation
but no Q&A logs.

Deterministic: same seed, same dataset.

In [ ]:
!cd {REPO}/dataset && python generate_raw.py

In [ ]:
import json
raw = [json.loads(l) for l in open(REPO / "dataset/raw/aura_support_raw.jsonl")]

print(f"{len(raw)} conversations\n")
r = raw[0]
print(f"topic: {r['topic']}")
for m in r["messages"]:
    print(f"\n  [{m['role']}]\n  {m['content'][:300]}")

## 3.3 The four formats

Now the part that matters. Same conversations, four encodings.

| format | shape | multi-turn? |
|---|---|---|
| **raw** | one text blob | only by convention |
| **Alpaca** | `instruction` / `input` / `output` | **no** |
| **ShareGPT** | `conversations[{from, value}]` | yes |
| **ChatML** | `messages[{role, content}]` | yes |

Run this and read the output — it explains the formats better than any
slide can.

In [ ]:
!cd {REPO}/dataset && python build_dataset.py --show 1

## 3.4 Where Alpaca breaks

Look at the multi-turn example above: **Alpaca dropped it.**

Alpaca has exactly one `output` field. A three-turn conversation has
nowhere to go. That's not a bug in our converter — it's the format's
actual limitation, and it's why the field moved to `messages`.

This matters practically: convert a chat dataset to Alpaca and you can
silently lose every multi-turn example without a single error.

In [ ]:
!cd {REPO}/dataset && python build_dataset.py

### A note on contamination

`build_dataset.py` also removes any training row phrased *exactly* like
one of the twelve evaluation questions.

Without that, Lab 4 would be a memorisation test — the model would recite
an answer it saw verbatim. Paraphrases of the same facts stay in, so it
still learns everything; it just has to generalise to the exact wording at
eval time. That's the honest version of the demo, and it's the same
discipline you'd want on a real project.

## 3.5 Load it as a `Dataset`

We train on the ChatML version. `standardize_sharegpt` exists to convert
ShareGPT's `from`/`value` naming into `role`/`content` — ours is already
in the target shape.

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "json",
    data_files = {
        "train": str(REPO / "dataset/formats/chatml_train.jsonl"),
        "val":   str(REPO / "dataset/formats/chatml_val.jsonl"),
    },
)
print(ds)
print("\nfirst example:")
for m in ds["train"][0]["messages"]:
    print(f"  [{m['role']:<9}] {m['content'][:110]}")

## 3.6 Apply the chat template — the step that silently ruins runs

In Lab 1 we saw the special tokens an instruct model expects. Training
data must carry **exactly** the same structure.

The failure mode: train with one template, infer with another. Nothing
errors. The model just underperforms, and you go looking for the problem
in your hyperparameters.

Unsloth's `get_chat_template` pins the tokenizer to a named template so
train and inference cannot drift apart.

> ↳ Slide: *Apply Chat Template*

In [ ]:
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
tokenizer = get_chat_template(tokenizer, chat_template=config.CHAT_TEMPLATE)

def formatting(batch):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]}

ds = ds.map(formatting, batched=True)

print("A formatted training example, exactly as the model will see it:")
print("=" * 70)
print(ds["train"][0]["text"][:900])
print("=" * 70)

## 3.7 Token lengths → choosing `max_seq_length`

`max_seq_length` is one of the biggest levers on training memory, because
attention activations scale with it.

Too low and long examples get truncated — often chopping the answer off a
training example, teaching the model to stop mid-sentence. Too high and
you waste VRAM padding.

Pick it from the data, not from a round number.

In [ ]:
lengths = [len(tokenizer(t)["input_ids"]) for t in ds["train"]["text"]]
lengths.sort()

def pct(p): return lengths[int(len(lengths) * p / 100)]

print(f"  examples     {len(lengths)}")
print(f"  min / max    {lengths[0]} / {lengths[-1]}")
print(f"  median       {pct(50)}")
print(f"  p90 / p99    {pct(90)} / {pct(99)}\n")

# Crude terminal histogram - readable without matplotlib.
buckets = [0] * 10
step = max(1, (lengths[-1] + 1) // 10)
for L in lengths:
    buckets[min(9, L // step)] += 1
for i, count in enumerate(buckets):
    bar = "#" * round(count / max(buckets) * 40)
    print(f"  {i*step:>5}-{(i+1)*step:<5} {count:>4} {bar}")

chosen = config.MAX_SEQ_LENGTH
over = sum(1 for L in lengths if L > chosen)
print(f"\n  max_seq_length = {chosen} truncates {over} examples "
      f"({over/len(lengths):.1%})")

## 3.8 `train_on_responses_only` — see the mask

By default the loss covers every token, so the model is partly trained to
*predict the user's questions*. That's wasted capacity: at inference the
question is given.

`train_on_responses_only` masks the prompt with `-100`, PyTorch's
"ignore this position" label. Loss is computed on the answer only.

The print below shows which tokens count. This is also the diagnostic for
`loss = 0.0` — if masking is misconfigured, everything gets ignored and the
loss goes to exactly zero.

In [ ]:
# Illustrative: this is what the trainer does internally in Lab 3.
sample = ds["train"][0]["text"]
ids = tokenizer(sample, return_tensors="pt")["input_ids"][0]

# Llama 3.x marks the assistant turn with this header.
marker = "<|start_header_id|>assistant<|end_header_id|>"
if marker not in sample:
    marker = "<|im_start|>assistant"          # Qwen / ChatML fallback

split_at = len(tokenizer(sample.split(marker)[0] + marker)["input_ids"])
labels = [-100] * split_at + ids[split_at:].tolist()

print(f"  {len(ids)} tokens, {split_at} masked, "
      f"{len(ids)-split_at} contribute to the loss\n")
print(f"  {'token':<22} {'label':>8}   counted?")
print("  " + "-" * 48)
for i in list(range(6)) + ["..."] + list(range(split_at-2, min(split_at+6, len(ids)))):
    if i == "...":
        print(f"  {'...':<22} {'...':>8}")
        continue
    tok = repr(tokenizer.decode([ids[i]]))[:20]
    counted = "no  (prompt)" if labels[i] == -100 else "YES (answer)"
    print(f"  {tok:<22} {labels[i]:>8}   {counted}")

## Ready to train

- **665** training conversations, **34** held out for validation
- ChatML format, chat template applied
- `max_seq_length` chosen from the actual distribution
- Loss will be computed on answers only

Next notebook is the core of the workshop.

---

### Next: `03_qlora_finetune.ipynb` — QLoRA — train 0.7% of the model

> **Kaggle tip:** if the session has been idle a while, check the right-hand
> panel still shows the GPU attached before starting the next notebook.